# revision_R1 production — GitHub-pinned runner

Cells A–G use one explicitly selected full Git commit per Colab session. Cell A fetches it once and checks out detached into a commit-versioned local directory, then verifies both manifests, the compatible training-core hash, and frozen schedule.

Drive data and durable files use Drive API v3 with the dedicated user-OAuth refresh token from Colab Secrets for the canonical main account. Drive v2, built from those exact same credentials, is used only for lease ETag/If-Match compare-and-swap because v3 no longer exposes file ETags. Cell A verifies both API identities and runs a live stale-write rejection probe before any run; missing ETags or a stale update not returning HTTP 412 keeps production blocked. It never uses `google.auth.default()` for storage. Cell B fetches the canonical preflight JSON, effective schedule, matching manifest, recovery reference, and uncached datasets by verified Drive file IDs into local staging. Verified mmap caches are reused. Cells A through C require no DriveFS mount. Every durable output, checkpoint, progress record, manifest, registry, lease, journal, and export remains staged locally and written through the verified main-account Drive v3 API.

Before Cell A, complete the one-time OAuth setup in `README_GITHUB_COLAB.md` and add `REVISION_R1_STORAGE_OAUTH_CLIENT_ID`, `REVISION_R1_STORAGE_OAUTH_CLIENT_SECRET`, and `REVISION_R1_STORAGE_REFRESH_TOKEN` to Colab Secrets with notebook access enabled. Never put secret values in notebook code or output.

The repository is currently public, so no GitHub token is required. If private later, use the fine-grained read-only Colab Secret `GITHUB_READ_TOKEN` (Contents: Read-only, restricted to this repo; enable notebook access). Never paste credentials into cells or outputs.

Run A → B → C → D. D does not change run state or locks; it writes its diagnostic report through the canonical API. B reads Drive datasets or checksum-verified local caches only; external downloads are disabled. Run 1 is already finalized/registered and is reused without remeasurement; E is a no-op. F remains disabled by default.

In [7]:
# Cell A — Drive/GitHub setup, one pinned detached fetch, and package integrity checks.
from pathlib import Path, PurePosixPath
import hashlib, json, os, platform, shutil, subprocess, sys, tempfile, urllib.error, urllib.request
from google.colab import userdata

GITHUB_REPOSITORY = "KianAnbarestani/AHR-MalCL-HR"
PINNED_GIT_COMMIT = "180d5511d8ad70ea040c61bf1f9c69e5aab37b59"  # Change only this full SHA for a tested code release.
GITHUB_TOKEN_SECRET_NAME = "GITHUB_READ_TOKEN"  # Used only if the repository is private.
STORAGE_ACCOUNT_EMAIL = "kian.anb777@gmail.com"
CANONICAL_REVISION_FOLDER_ID = "1tQgfvJNt5MX4cNQbH-4U30hJpcDrBO8x"
EXPECTED_TRAINING_SOURCE_SHA256 = "1731ee525e3297e276609040295977e4c02f5aeab8e4ab125dd08fbcb993dc9c"
EXPECTED_SCHEDULE_SHA256 = "7d327c9ce84a096201d06f910f88d022f4e056ad32e090a84e19be393747d212"
if len(PINNED_GIT_COMMIT) != 40 or any(c not in "0123456789abcdef" for c in PINNED_GIT_COMMIT):
    raise ValueError("PINNED_GIT_COMMIT must be the explicitly selected full lowercase 40-character commit SHA")

DRIVE_PATHS = {
    "ember_data_root": "/content/drive/MyDrive/AHR_MalCL/datasets",
    "az_data_root": "/content/drive/MyDrive/AHR_MalCL/datasets",
    "preflight_drive_path": "/content/drive/MyDrive/ReVision/revision_R1/results/preflight/preflight.json",
    "preflight_staging_root": "/content/revision_R1_storage_staging/preflight",
    "v62_recovery_report": "/content/drive/MyDrive/ReVision/revision_R1/results/validation/reports/resume_validation_2b4d8f5f1f29a938_20261026T142320Z_946699443.json",
    "output_root": "/content/drive/MyDrive/ReVision/revision_R1/production/results",
    "checkpoint_root": "/content/drive/MyDrive/ReVision/revision_R1/production/checkpoints",
    "dataset_cache_root": "/content/revision_R1_production/cache",
    "local_checkpoint_root": "/content/revision_R1_production/checkpoints",
    "export_root": "/content/drive/MyDrive/ReVision/revision_R1/production/exports",
    "checkpoint_backup_root": "/content/drive/MyDrive/ReVision/revision_R1/production/checkpoint_backups",
}

GITHUB_API_URL = f"https://api.github.com/repos/{GITHUB_REPOSITORY}"
def _github_metadata(token=None):
    headers = {"Accept": "application/vnd.github+json", "X-GitHub-Api-Version": "2022-11-28"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    request = urllib.request.Request(GITHUB_API_URL, headers=headers)
    try:
        with urllib.request.urlopen(request, timeout=20) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        if not token and exc.code == 404:
            return None  # GitHub hides private repositories from unauthenticated callers.
        raise RuntimeError(f"GitHub repository access failed (HTTP {exc.code}); token contents are never displayed") from None
    except (OSError, json.JSONDecodeError):
        raise RuntimeError("Could not verify GitHub repository visibility/identity; check network and retry Cell A") from None

REPOSITORY_METADATA = _github_metadata()
GITHUB_TOKEN = None
if REPOSITORY_METADATA is None or REPOSITORY_METADATA.get("private") is True:
    try:
        from google.colab import userdata
        GITHUB_TOKEN = userdata.get(GITHUB_TOKEN_SECRET_NAME)
    except Exception:
        GITHUB_TOKEN = None
    if not GITHUB_TOKEN:
        raise RuntimeError(
            f"This repository is private or not publicly resolvable. Add a fine-grained read-only GitHub token "
            f"to Colab Secrets as {GITHUB_TOKEN_SECRET_NAME}, restrict it to {GITHUB_REPOSITORY} with Contents: Read-only, "
            "and enable notebook access. Do not paste it into a cell."
        )
    try:
        REPOSITORY_METADATA = _github_metadata(GITHUB_TOKEN)
    except Exception:
        GITHUB_TOKEN = None
        raise
if REPOSITORY_METADATA.get("full_name", "").lower() != GITHUB_REPOSITORY.lower():
    GITHUB_TOKEN = None
    raise RuntimeError("GitHub returned a different repository identity than the pinned notebook setting")

CHECKOUT_PARENT = Path("/content/revision_R1_pinned_sources")
CHECKOUT_PATH = CHECKOUT_PARENT / PINNED_GIT_COMMIT
PACKAGE_ROOT = CHECKOUT_PATH / "revision_R1"
def _git(args, *, cwd, env):
    result = subprocess.run(["git", *args], cwd=cwd, env=env, text=True, capture_output=True, check=False)
    if result.returncode:
        # Raw stderr is intentionally withheld to prevent accidental credential disclosure.
        raise RuntimeError(f"Git operation {args[0]!r} failed (exit {result.returncode}); verify the commit and repository read access")
    return result.stdout.strip()

if CHECKOUT_PATH.exists():
    GITHUB_TOKEN = None  # Existing verified checkout needs no credential for local reuse.
    if not (CHECKOUT_PATH / ".git").is_dir():
        raise RuntimeError(f"Pinned checkout path exists but is not a Git checkout; preserved: {CHECKOUT_PATH}")
    head = _git(["rev-parse", "HEAD"], cwd=CHECKOUT_PATH, env=os.environ.copy())
    if head != PINNED_GIT_COMMIT:
        raise RuntimeError(f"Existing session checkout is a different commit; preserved: {head}")
    if _git(["status", "--porcelain"], cwd=CHECKOUT_PATH, env=os.environ.copy()):
        raise RuntimeError("Pinned session checkout is modified; refusing to reuse a dirty checkout")
else:
    CHECKOUT_PARENT.mkdir(parents=True, exist_ok=True)
    staging = Path(tempfile.mkdtemp(prefix=f".{PINNED_GIT_COMMIT[:12]}-", dir=CHECKOUT_PARENT))
    askpass_dir = None
    token = GITHUB_TOKEN
    git_env = os.environ.copy()
    git_env.update({"GIT_TERMINAL_PROMPT": "0", "GIT_TRACE": "0", "GIT_CURL_VERBOSE": "0",
                    "GIT_CONFIG_NOSYSTEM": "1", "GIT_CONFIG_GLOBAL": os.devnull})
    if token:
        askpass_dir = tempfile.TemporaryDirectory(prefix="revision_r1_git_auth_")
        askpass = Path(askpass_dir.name) / "askpass.py"
        askpass.write_text(
            "#!/usr/bin/env python3\nimport os,sys\np=sys.argv[-1].lower()\n"
            "print('x-access-token' if 'username' in p else os.environ.get('GITHUB_READ_TOKEN',''))\n",
            encoding="utf-8",
        )
        askpass.chmod(0o700)
        git_env.update({"GIT_ASKPASS": str(askpass), "GIT_ASKPASS_REQUIRE": "force", "GITHUB_READ_TOKEN": token})
    try:
        _git(["init", "--quiet"], cwd=staging, env=git_env)
        _git(["remote", "add", "origin", f"https://github.com/{GITHUB_REPOSITORY}.git"], cwd=staging, env=git_env)
        _git(["fetch", "--no-tags", "--depth=1", "origin", PINNED_GIT_COMMIT], cwd=staging, env=git_env)
        _git(["checkout", "--detach", "FETCH_HEAD"], cwd=staging, env=git_env)
        fetched = _git(["rev-parse", "HEAD"], cwd=staging, env=git_env)
        if fetched != PINNED_GIT_COMMIT:
            raise RuntimeError("Fetched object did not match PINNED_GIT_COMMIT")
        if _git(["remote", "get-url", "origin"], cwd=staging, env=git_env) != f"https://github.com/{GITHUB_REPOSITORY}.git":
            raise RuntimeError("Git remote is not credential-free; refusing to retain it")
        os.replace(staging, CHECKOUT_PATH)
    finally:
        if staging.exists():
            shutil.rmtree(staging, ignore_errors=True)
        if askpass_dir is not None:
            askpass_dir.cleanup()
        git_env.pop("GITHUB_READ_TOKEN", None)
        GITHUB_TOKEN = None
        token = None

ACTUAL_GIT_COMMIT = _git(["rev-parse", "HEAD"], cwd=CHECKOUT_PATH, env=os.environ.copy())
if ACTUAL_GIT_COMMIT != PINNED_GIT_COMMIT or _git(["rev-parse", "--abbrev-ref", "HEAD"], cwd=CHECKOUT_PATH, env=os.environ.copy()) != "HEAD":
    raise RuntimeError("Pinned source was not checked out at the exact detached commit")

def _sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

PROJECT = PACKAGE_ROOT
os.chdir(PROJECT)
sys.dont_write_bytecode = True
sys.path[:] = [entry for entry in sys.path if Path(entry or ".").resolve() != PROJECT.resolve()]
sys.path.insert(0, str(PROJECT))
for module_name in list(sys.modules):
    if module_name == "revision_r1" or module_name.startswith("revision_r1.") or module_name == "scripts" or module_name.startswith("scripts."):
        del sys.modules[module_name]
from scripts.build_github_handoff import verify_package_integrity
PACKAGE_MANIFEST, BUILD_MANIFEST, PACKAGE_MANIFEST_SHA256 = verify_package_integrity(PACKAGE_ROOT)
from scripts.build_production_handoff import PRODUCTION_BUILD_ID as EXPECTED_BUILD_ID
import numpy as np, platform, torch
EXPECTED_PYTHON_VERSION = (3, 12, 13)
EXPECTED_TORCH_VERSION = "2.11.0+cu128"
EXPECTED_CUDA_VERSION = "12.8"
if sys.version_info[:3] != EXPECTED_PYTHON_VERSION:
    raise RuntimeError(f"Expected Colab Python {'.'.join(map(str, EXPECTED_PYTHON_VERSION))}, got {platform.python_version()}")
if np.__version__ != "2.0.2":
    raise RuntimeError(f"Expected pinned NumPy 2.0.2, got {np.__version__}; no packages were changed")
if torch.__version__ != EXPECTED_TORCH_VERSION:
    raise RuntimeError(f"Expected preinstalled CUDA PyTorch {EXPECTED_TORCH_VERSION}, got {torch.__version__}; no packages were changed")
if torch.version.cuda != EXPECTED_CUDA_VERSION:
    raise RuntimeError(f"Expected PyTorch CUDA build {EXPECTED_CUDA_VERSION}, got {torch.version.cuda}; no packages were changed")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. No training started; select a GPU Colab runtime and rerun Cell A.")
from scripts.production_workflow import training_source_sha256
from revision_r1.compatibility import runtime_source_sha256, verify_checkpoint_compatibility
import revision_r1.runner as runner_module
import scripts.production_workflow as production_workflow_module
# Authenticate storage only after fetching and verifying the pinned package that
# supplies this helper. It uses explicit OAuth secrets; cached Colab ADC is never read.
print(f"Canonical Drive storage account: {STORAGE_ACCOUNT_EMAIL}; verifying dedicated OAuth credentials.")
from scripts.storage_auth import build_main_account_storage_api
STORAGE_API_SERVICE, STORAGE_CREDENTIALS, STORAGE_AUTH_EVIDENCE = build_main_account_storage_api(
    secret_getter=userdata.get, expected_account_email=STORAGE_ACCOUNT_EMAIL,
)
STORAGE_API_FACTORY = lambda: STORAGE_API_SERVICE
print("Drive API OAuth identity verified:", STORAGE_AUTH_EVIDENCE["authenticated_user"],
      "credential source:", STORAGE_AUTH_EVIDENCE["credential_source"])
# Drive paths below are logical API addresses; setup requires no DriveFS mount.
ACTUAL_TRAINING_SOURCE_SHA256 = training_source_sha256(PROJECT)
CHECKPOINT_COMPATIBILITY = verify_checkpoint_compatibility(PROJECT)
ACTUAL_RUNTIME_SOURCE_SHA256 = runtime_source_sha256(PROJECT)
if (ACTUAL_TRAINING_SOURCE_SHA256 != EXPECTED_TRAINING_SOURCE_SHA256
    or runner_module._code_sha256() != EXPECTED_TRAINING_SOURCE_SHA256
    or PACKAGE_MANIFEST.get("training_source_sha256") != ACTUAL_TRAINING_SOURCE_SHA256
    or PACKAGE_MANIFEST.get("runtime_source_sha256") != ACTUAL_RUNTIME_SOURCE_SHA256):
    raise ValueError("Training-core hash is incompatible with the resumable first-run checkpoint; no fallback/restart is allowed")
if not Path(runner_module.__file__).resolve().is_relative_to(PROJECT.resolve()):
    raise ImportError(f"Training runner imported outside the pinned Git checkout: {runner_module.__file__}")
from revision_r1.drive_storage import DriveApiStorage
STORAGE_STAGE_ROOT = Path("/content/revision_R1_storage_staging").resolve()
STORAGE_ADAPTER = DriveApiStorage(
    service=STORAGE_API_SERVICE, expected_account_email=STORAGE_ACCOUNT_EMAIL,
    revision_folder_id=CANONICAL_REVISION_FOLDER_ID,
    logical_revision_root="/content/drive/MyDrive/ReVision",
    local_staging_root=STORAGE_STAGE_ROOT,
)
STORAGE_ADAPTER.bind_root(
    Path(DRIVE_PATHS["preflight_drive_path"]).parent,
    DRIVE_PATHS["preflight_staging_root"],
)
STORAGE_ADAPTER.bind_root(DRIVE_PATHS["output_root"], STORAGE_STAGE_ROOT / "production/results")
STORAGE_ADAPTER.bind_root(DRIVE_PATHS["checkpoint_root"], STORAGE_STAGE_ROOT / "production/checkpoints")
STORAGE_ADAPTER.bind_root(DRIVE_PATHS["export_root"], STORAGE_STAGE_ROOT / "production/exports")
STORAGE_ADAPTER.bind_root(DRIVE_PATHS["checkpoint_backup_root"], STORAGE_STAGE_ROOT / "production/checkpoint_backups")
try:
    STORAGE_SESSION_EVIDENCE = production_workflow_module.verify_canonical_storage_session(
        api_factory=STORAGE_API_FACTORY,
        drive_mount_root="/content/drive/MyDrive",
        expected_account_email=STORAGE_ACCOUNT_EMAIL,
        expected_revision_folder_id=CANONICAL_REVISION_FOLDER_ID,
        storage_adapter=STORAGE_ADAPTER,
        path_bindings={
            "revision_root": "/content/drive/MyDrive/ReVision",
            "datasets": "/content/drive/MyDrive/AHR_MalCL/datasets",
            "preflight": "/content/drive/MyDrive/ReVision/revision_R1/results/preflight",
            "validation_reports": "/content/drive/MyDrive/ReVision/revision_R1/results/validation/reports",
            "output_root": "/content/drive/MyDrive/ReVision/revision_R1/production/results",
            "checkpoint_root": "/content/drive/MyDrive/ReVision/revision_R1/production/checkpoints",
            "production_results": "/content/drive/MyDrive/ReVision/revision_R1/production/results",
            "production_checkpoints": "/content/drive/MyDrive/ReVision/revision_R1/production/checkpoints",
        },
        required_path_keys=(
            "revision_root", "datasets", "preflight", "validation_reports",
            "output_root", "checkpoint_root", "production_results", "production_checkpoints",
        ),
    )
except Exception as exc:
    raise RuntimeError(
        "Canonical Drive storage verification failed before any run. Durable storage must use the explicit OAuth "
        f"account {STORAGE_ACCOUNT_EMAIL} and ReVision folder {CANONICAL_REVISION_FOLDER_ID}; the compute/mount "
        "account may differ and is used only for reading datasets/preflight. Check the three REVISION_R1_STORAGE_* Colab "
        "Secrets, then rerun Cell A; do not proceed or change "
        f"paths. Cause: {type(exc).__name__}: {exc}"
    ) from exc
print(json.dumps({"repository": GITHUB_REPOSITORY, "repository_private": bool(REPOSITORY_METADATA.get("private")),
                  "selected_git_commit": ACTUAL_GIT_COMMIT, "checkout": str(CHECKOUT_PATH),
                  "detached_checkout": True, "package_manifest_sha256": PACKAGE_MANIFEST_SHA256,
                  "training_source_sha256": ACTUAL_TRAINING_SOURCE_SHA256,
                  "runtime_source_sha256": ACTUAL_RUNTIME_SOURCE_SHA256,
                  "effective_schedule_sha256": EXPECTED_SCHEDULE_SHA256,
                  "python": platform.python_version(), "numpy": np.__version__, "torch": torch.__version__,
                  "gpu": torch.cuda.get_device_name(0),
                  "storage_account": STORAGE_SESSION_EVIDENCE["storage_account_email"],
                  "storage_quota_account": STORAGE_SESSION_EVIDENCE["storage_quota_account_email"],
                  "canonical_revision_folder_id": STORAGE_SESSION_EVIDENCE["canonical_revision_folder_id"],
                  "durable_storage_backend": "explicit_main_account_drive_api_with_local_staging",
                  "compute_mount_identity_used_for_durable_writes": False}, indent=2))


Canonical Drive storage account: kian.anb777@gmail.com; verifying dedicated OAuth credentials.
Drive API OAuth identity verified: kian.anb777@gmail.com credential source: explicit_colab_secret_user_oauth
{
  "repository": "KianAnbarestani/AHR-MalCL-HR",
  "repository_private": false,
  "selected_git_commit": "180d5511d8ad70ea040c61bf1f9c69e5aab37b59",
  "checkout": "/content/revision_R1_pinned_sources/180d5511d8ad70ea040c61bf1f9c69e5aab37b59",
  "detached_checkout": true,
  "package_manifest_sha256": "d01aaa9b3dcb36cbe0d3d5988c464ed54246a23c0d1ff69cb83576d3e31a42c2",
  "training_source_sha256": "1731ee525e3297e276609040295977e4c02f5aeab8e4ab125dd08fbcb993dc9c",
  "runtime_source_sha256": "e41fd7512a1404a9b244cf492a10a8ae27bd5e976300f36c8b4077299815303d",
  "effective_schedule_sha256": "7d327c9ce84a096201d06f910f88d022f4e056ad32e090a84e19be393747d212",
  "python": "3.12.13",
  "numpy": "2.0.2",
  "torch": "2.11.0+cu128",
  "gpu": "Tesla T4",
  "storage_account": "kian.anb777@gmail.com",

In [8]:
# Cell B — OAuth API preflight/data staging; no training or preflight regeneration.
from pathlib import Path
import hashlib, json, os
from revision_r1.validation_workflow import resolve_effective_artifacts
from revision_r1.config import DATASETS
from revision_r1.data import inspect_npz_array_layout, file_digest
from revision_r1.drive_storage import DriveApiStorage
from scripts.production_workflow import (
    inspect_dataset_sources, resolve_drive_folder,
    _verified_cache_candidate, _load_verified_cache_split,
)
from scripts.build_github_handoff import stage_canonical_preflight_artifacts

STORAGE_STAGE_ROOT = Path(STORAGE_ADAPTER.local_staging_root)
DRIVE_PATHS.setdefault("preflight_drive_path", str(
    STORAGE_ADAPTER.logical_revision_root / "revision_R1/results/preflight/preflight.json"))
DRIVE_PATHS.setdefault("preflight_staging_root", str(STORAGE_STAGE_ROOT / "preflight"))
PREFLIGHT_STAGING_EVIDENCE = stage_canonical_preflight_artifacts(
    STORAGE_ADAPTER,
    DRIVE_PATHS["preflight_drive_path"],
    DRIVE_PATHS["preflight_staging_root"],
    EXPECTED_SCHEDULE_SHA256,
)
PREFLIGHT_PATH = Path(PREFLIGHT_STAGING_EVIDENCE["preflight_path"]).expanduser()
if not PREFLIGHT_PATH.is_file() or PREFLIGHT_PATH.is_symlink():
    raise FileNotFoundError(f"OAuth API-staged preflight is missing or unsafe: {PREFLIGHT_PATH}")
EFFECTIVE_SCHEDULE_PATH, EFFECTIVE_SCHEDULE_MANIFEST_PATH, PREFLIGHT = resolve_effective_artifacts(
    PREFLIGHT_PATH, expected_sha256=EXPECTED_SCHEDULE_SHA256, discover_manifest=True,
)
if (EFFECTIVE_SCHEDULE_PATH != Path(PREFLIGHT_STAGING_EVIDENCE["schedule_path"])
    or EFFECTIVE_SCHEDULE_MANIFEST_PATH != Path(PREFLIGHT_STAGING_EVIDENCE["manifest_path"])
    or PREFLIGHT_STAGING_EVIDENCE["schedule_sha256"] != EXPECTED_SCHEDULE_SHA256):
    raise ValueError("Cell B resolved preflight artifacts differ from their verified canonical Drive IDs")
if PREFLIGHT.get("status") != "PREFLIGHT_VALIDATED_NO_TRAINING" or PREFLIGHT.get("training_reproduction_claimed") is not False:
    raise ValueError("Drive preflight is not the validated no-training schedule/data record")
DRIVE_PATHS["preflight_path"] = str(PREFLIGHT_PATH)
DRIVE_PATHS.setdefault("dataset_drive_roots", {
    "ember": str(DRIVE_PATHS["ember_data_root"]),
    "az_class": str(DRIVE_PATHS["az_data_root"]),
})
LOCAL_DATASET_CACHE_ROOT = Path(DRIVE_PATHS["dataset_cache_root"]).expanduser().resolve()
DATA_ROOTS = {name: STORAGE_STAGE_ROOT / "dataset_sources" / name for name in DATASETS}
DATASET_API_STAGING_EVIDENCE = {}
for _dataset, _root in DATA_ROOTS.items():
    _reader = None
    _logical_root = Path(DRIVE_PATHS["dataset_drive_roots"][_dataset]).expanduser()
    for _split in ("train", "test"):
        _spec = DATASETS[_dataset][_split]
        _expected = PREFLIGHT[_dataset][_split]
        if (_expected["md5"] != _spec["md5"] or len(_expected["sha256"]) != 64):
            raise ValueError(f"Dataset preflight identity differs from frozen protocol: {_dataset}/{_split}")
        _key = f"{_dataset}/{_split}"
        if _verified_cache_candidate(LOCAL_DATASET_CACHE_ROOT, _dataset, _split, PREFLIGHT) is not None:
            # Fully verify mmap contents; no archive download or decompression.
            _x, _y, _metadata = _load_verified_cache_split(
                LOCAL_DATASET_CACHE_ROOT, _dataset, _split, PREFLIGHT)
            del _x, _y
            DATASET_API_STAGING_EVIDENCE[_key] = {"source": "VERIFIED_LOCAL_MMAP_CACHE"}
            print("Dataset:", _key, "verified mmap cache reused", flush=True)
            continue
        _target = _root / _spec["filename"]
        if _target.exists() or _target.is_symlink():
            if (_target.is_symlink() or not _target.is_file()
                or file_digest(_target, "sha256") != _expected["sha256"]
                or file_digest(_target, "md5") != _expected["md5"]):
                raise ValueError(f"Existing local dataset staging differs; preserved: {_target}")
            DATASET_API_STAGING_EVIDENCE[_key] = {"source": "VERIFIED_LOCAL_NPZ", "path": str(_target)}
            continue
        if _reader is None:
            try:
                _components = _logical_root.relative_to("/content/drive/MyDrive").parts
            except ValueError as _error:
                raise ValueError("Dataset API source must be a configured logical MyDrive path") from _error
            _folder = resolve_drive_folder(STORAGE_ADAPTER.service, _components)
            if not _folder["path_exists"] or not _folder["folder_id"]:
                raise FileNotFoundError(f"Existing dataset Drive folder is absent: {_logical_root}")
            _reader = DriveApiStorage(
                service=STORAGE_ADAPTER.service,
                expected_account_email=STORAGE_ADAPTER.expected_account_email,
                revision_folder_id=_folder["folder_id"],
                logical_revision_root=_logical_root, local_staging_root=_root,
                download_to_path=STORAGE_ADAPTER.download_to_path,
            )
        _logical_file = _logical_root / _spec["filename"]
        _remote = _reader.exact_file(_logical_file)
        if _remote is None or _remote.get("md5Checksum") != _expected["md5"]:
            raise ValueError(f"Missing or mismatched canonical dataset API source: {_logical_file}")
        print("Dataset:", _key, "API download starting; bytes:", _remote.get("size"), flush=True)
        _evidence = _reader.download_file(
            _logical_file, _target, expected_sha256=_expected["sha256"],
            expected_bytes=int(_remote["size"]),
        )
        if file_digest(_target, "md5") != _expected["md5"]:
            raise ValueError(f"Dataset download MD5 differs from frozen identity: {_target}")
        DATASET_API_STAGING_EVIDENCE[_key] = {"source": "CANONICAL_OAUTH_API", **_evidence}
        print("Dataset:", _key, "API download verified", flush=True)
# The runner builds/reuses its streamed mmap cache once when each dataset is
# first used. This cell only inspects NPZ headers; it never extracts arrays.
DRIVE_PATHS["ember_data_root"] = str(DATA_ROOTS["ember"])
DRIVE_PATHS["az_data_root"] = str(DATA_ROOTS["az_class"])
DATASET_SOURCE_REPORT = inspect_dataset_sources(DATA_ROOTS, LOCAL_DATASET_CACHE_ROOT, PREFLIGHT)
DATASET_LAYOUTS = {}
for _dataset, _root in DATA_ROOTS.items():
    _layout = {}
    for _split in ("train", "test"):
        _npz = _root / DATASETS[_dataset][_split]["filename"]
        if _npz.is_file():
            _layout[_split] = inspect_npz_array_layout(_npz, split=_split)
        else:
            _layout[_split] = {"archive": "ABSENT", "source": DATASET_SOURCE_REPORT[_dataset]["splits"][_split]["source"]}
    DATASET_LAYOUTS[_dataset] = _layout

# Cell C also consumes this existing small report; stage it through OAuth so
# the readiness interface has no hidden DriveFS dependency.
DRIVE_PATHS.setdefault("v62_recovery_report_drive_path", DRIVE_PATHS["v62_recovery_report"])
_reference_logical = Path(DRIVE_PATHS["v62_recovery_report_drive_path"])
_reference_relative = _reference_logical.relative_to(STORAGE_ADAPTER.logical_revision_root)
_reference_local = STORAGE_STAGE_ROOT / "read_only_references" / _reference_relative
REFERENCE_STAGING_EVIDENCE = STORAGE_ADAPTER.download_file(_reference_logical, _reference_local)
DRIVE_PATHS["v62_recovery_report"] = str(_reference_local)

PACKAGED_SCHEDULE_PATH = PROJECT / "configs" / "update_budget.csv"
PACKAGED_SCHEDULE_MANIFEST_PATH = PROJECT / "configs" / "update_budget_manifest.json"
PACKAGED_SCHEDULE_SHA256 = hashlib.sha256(PACKAGED_SCHEDULE_PATH.read_bytes()).hexdigest()
PACKAGED_SCHEDULE_MANIFEST = json.loads(PACKAGED_SCHEDULE_MANIFEST_PATH.read_text(encoding="utf-8"))
if PACKAGED_SCHEDULE_SHA256 != PACKAGED_SCHEDULE_MANIFEST.get("source_csv_sha256"):
    raise ValueError("Original packaged schedule/manifest pair is altered")
print("Effective schedule:", EFFECTIVE_SCHEDULE_PATH)
print("Effective schedule SHA-256:", EXPECTED_SCHEDULE_SHA256, "| rows:", PREFLIGHT["schedule"]["row_count"])
print("Matching manifest:", EFFECTIVE_SCHEDULE_MANIFEST_PATH)
print("Canonical OAuth API preflight artifact IDs and local staging:")
print(json.dumps(PREFLIGHT_STAGING_EVIDENCE, sort_keys=True, indent=2))
print("Original packaged schedule SHA-256 (unchanged):", PACKAGED_SCHEDULE_SHA256)
print("Canonical dataset sources (missing Drive files require a matching local verified cache):")
print(json.dumps(DATASET_SOURCE_REPORT, sort_keys=True, indent=2))
print("Dataset API/cache staging:", json.dumps(DATASET_API_STAGING_EVIDENCE, sort_keys=True))
print("If both canonical Drive files and a matching verified local cache are absent, preflight fails; no external download will be attempted.")
print("External dataset downloads: DISABLED")


Effective schedule: /content/revision_R1_storage_staging/preflight/6be7ccaf4c1f77c5aab9a3ff450e8739b733b0002a6785cfd218ac2a830811a9/schedules/effective_update_budget_b7d672906463_7d327c9ce84a.csv
Effective schedule SHA-256: 7d327c9ce84a096201d06f910f88d022f4e056ad32e090a84e19be393747d212 | rows: 220
Matching manifest: /content/revision_R1_storage_staging/preflight/6be7ccaf4c1f77c5aab9a3ff450e8739b733b0002a6785cfd218ac2a830811a9/schedules/effective_update_budget_b7d672906463_7d327c9ce84a.manifest.json
Canonical OAuth API preflight artifact IDs and local staging:
{
  "bundle_key": "6be7ccaf4c1f77c5aab9a3ff450e8739b733b0002a6785cfd218ac2a830811a9",
  "canonical_preflight_manifest_path": "/content/drive/MyDrive/ReVision/revision_R1/results/preflight/schedules/effective_update_budget_b7d672906463_7d327c9ce84a.manifest.json",
  "canonical_preflight_schedule_path": "/content/drive/MyDrive/ReVision/revision_R1/results/preflight/schedules/effective_update_budget_b7d672906463_7d327c9ce84a.csv",


In [9]:
# Cell C — API-backed local staging, finalized Run 1 reuse, and storage readiness only.
from scripts.production_workflow import (
    _expected_plan_runner_identity, _preflight_addressed_cache_bytes, _read_matrix_registry,
    _hydrate_api_matrix_registry, _hydrate_api_run_state, matrix_registry_evidence,
    bind_runtime_schedule, build_frozen_run_plan, inspect_production_capacity,
    load_v62_recovery_checkpoint_bound, project_storage, require_run_capacity,
    reuse_validated_first_run_measurement, run_checkpoint_root, run_output_root,
    validate_frozen_plan, validate_runtime_bound_plan,
)
from revision_r1.storage import DEFAULT_STORAGE_RESERVE_BYTES, require_available_bytes
import os

BASE_PLAN_PATH = PROJECT / "configs" / "production_run_plan_v1.json"
BASE_PLAN = json.loads(BASE_PLAN_PATH.read_text(encoding="utf-8"))
validate_frozen_plan(BASE_PLAN, expected_source_sha256=ACTUAL_TRAINING_SOURCE_SHA256)
RUNTIME_PLAN = bind_runtime_schedule(BASE_PLAN, EFFECTIVE_SCHEDULE_PATH, EFFECTIVE_SCHEDULE_MANIFEST_PATH, PREFLIGHT)
validate_runtime_bound_plan(RUNTIME_PLAN, EFFECTIVE_SCHEDULE_PATH, EFFECTIVE_SCHEDULE_MANIFEST_PATH, PREFLIGHT)
if RUNTIME_PLAN["run_count"] != 100 or RUNTIME_PLAN["runs"][0]["run_key"] != "ember_seed42_full_hybrid":
    raise ValueError("Runtime-bound matrix/order differs from the frozen protocol")

LOGICAL_OUTPUT_ROOT = Path(DRIVE_PATHS["output_root"]).expanduser()
LOGICAL_CHECKPOINT_ROOT = Path(DRIVE_PATHS["checkpoint_root"]).expanduser()
OUTPUT_ROOT = STORAGE_STAGE_ROOT / "production/results"
CHECKPOINT_ROOT = STORAGE_STAGE_ROOT / "production/checkpoints"
LOCAL_CHECKPOINT_ROOT = Path(DRIVE_PATHS["local_checkpoint_root"]).expanduser()
CACHE_ROOT = Path(DRIVE_PATHS["dataset_cache_root"]).expanduser()
EXPORT_ROOT = STORAGE_STAGE_ROOT / "production/exports"
CHECKPOINT_BACKUP_ROOT = STORAGE_STAGE_ROOT / "production/checkpoint_backups"
FIRST_RUN = RUNTIME_PLAN["runs"][0]
_hydrate_api_matrix_registry(STORAGE_ADAPTER, OUTPUT_ROOT, RUNTIME_PLAN)
_hydrate_api_run_state(
    STORAGE_ADAPTER, OUTPUT_ROOT, RUNTIME_PLAN, FIRST_RUN,
    run_output_root(OUTPUT_ROOT, FIRST_RUN), run_checkpoint_root(CHECKPOINT_ROOT, FIRST_RUN),
    include_recovery=False,
)
MATRIX_REGISTRY_EVIDENCE = matrix_registry_evidence(STORAGE_ADAPTER, OUTPUT_ROOT, RUNTIME_PLAN)
print("CELL_C_CANONICAL_COMPLETION_REGISTRY:", json.dumps(MATRIX_REGISTRY_EVIDENCE, sort_keys=True))
FIRST_RUN_MEASUREMENT = reuse_validated_first_run_measurement(RUNTIME_PLAN, FIRST_RUN, OUTPUT_ROOT)
MATRIX_REGISTRY = _read_matrix_registry(OUTPUT_ROOT.resolve(), RUNTIME_PLAN)
if FIRST_RUN["run_key"] not in MATRIX_REGISTRY["completed_runs"]:
    raise RuntimeError("Run 1 is not registered complete; this release will not launch or finalize it")
COMPLETED_RUN_KEYS = set(MATRIX_REGISTRY["completed_runs"])
NEXT_PENDING_RECORD = next((row for row in RUNTIME_PLAN["runs"][1:] if row["run_key"] not in COMPLETED_RUN_KEYS), None)
V62_REFERENCE = load_v62_recovery_checkpoint_bound(DRIVE_PATHS["v62_recovery_report"])
OBSERVED_CHECKPOINT_PAYLOAD_BYTES = max(
    int(FIRST_RUN_MEASUREMENT["max_checkpoint_payload_bytes"]),
    int(V62_REFERENCE["max_checkpoint_payload_bytes"]),
)
STORAGE_PROJECTION = project_storage(
    RUNTIME_PLAN,
    observed_checkpoint_bytes=OBSERVED_CHECKPOINT_PAYLOAD_BYTES,
    observed_output_bytes=int(FIRST_RUN_MEASUREMENT["run_output_bytes"]),
    checkpoint_reference_provenance=(
        "registered Run 1 measurement reused plus the verified v6.2 operational reference; "
        "planning estimate only, not a new measurement"
    ),
    checkpoint_size_evidence={
        "run_1_registered_measurement_reused": FIRST_RUN_MEASUREMENT,
        "v6_2_reference_payload_bytes": int(V62_REFERENCE["max_checkpoint_payload_bytes"]),
        "projection_source_bytes": OBSERVED_CHECKPOINT_PAYLOAD_BYTES,
    },
)
if NEXT_PENDING_RECORD is None:
    CAPACITY = inspect_production_capacity(
        LOGICAL_OUTPUT_ROOT, LOGICAL_CHECKPOINT_ROOT, LOCAL_CHECKPOINT_ROOT, CACHE_ROOT,
        api_factory=STORAGE_API_FACTORY, storage_adapter=STORAGE_ADAPTER,
    )
else:
    NEXT_OUTPUT = run_output_root(OUTPUT_ROOT, NEXT_PENDING_RECORD)
    NEXT_CHECKPOINT = run_checkpoint_root(CHECKPOINT_ROOT, NEXT_PENDING_RECORD)
    NEXT_LOCAL_CHECKPOINT = run_checkpoint_root(LOCAL_CHECKPOINT_ROOT, NEXT_PENDING_RECORD)
    CAPACITY = inspect_production_capacity(
        LOGICAL_OUTPUT_ROOT, LOGICAL_CHECKPOINT_ROOT, LOCAL_CHECKPOINT_ROOT, CACHE_ROOT,
        api_factory=STORAGE_API_FACTORY, storage_adapter=STORAGE_ADAPTER,
        current_run_output_path=NEXT_OUTPUT,
        current_run_checkpoint_path=NEXT_CHECKPOINT,
        current_run_local_checkpoint_path=NEXT_LOCAL_CHECKPOINT,
        expected_run_identity=_expected_plan_runner_identity(RUNTIME_PLAN, NEXT_PENDING_RECORD),
        verified_dataset_cache_bytes=_preflight_addressed_cache_bytes(
            PREFLIGHT, CACHE_ROOT, dataset=NEXT_PENDING_RECORD["dataset"],
        ),
    )
    CAPACITY["production_measurements_validated"] = True
    require_run_capacity(
        CAPACITY, STORAGE_PROJECTION, scope="matrix",
        completed_runs=len(MATRIX_REGISTRY["completed_runs"]),
    )
if CAPACITY.get("canonical_storage_identity_verified") is not True:
    raise RuntimeError("Capacity evidence is not bound to the canonical main-account storage session")

# Bind selected release separately; neither compute account nor Git commit changes training identity.
CODE_BINDING_PATH = OUTPUT_ROOT / "runtime_code_binding" / RUNTIME_PLAN["plan_sha256"] / f"{ACTUAL_GIT_COMMIT}.json"
CODE_BINDING = {
    "schema_version": 1, "selected_git_commit": ACTUAL_GIT_COMMIT,
    "package_manifest_sha256": PACKAGE_MANIFEST_SHA256,
    "training_source_sha256": ACTUAL_TRAINING_SOURCE_SHA256,
    "runtime_source_sha256": ACTUAL_RUNTIME_SOURCE_SHA256,
    "effective_schedule_sha256": EXPECTED_SCHEDULE_SHA256,
    "frozen_plan_sha256": RUNTIME_PLAN["plan_sha256"],
    "first_run_key": FIRST_RUN["run_key"], "first_run_config_sha256": FIRST_RUN["config_sha256"],
}
CODE_BINDING_BYTES = (json.dumps(CODE_BINDING, sort_keys=True, indent=2, allow_nan=False) + "\n").encode("utf-8")
PLAN_PATH = OUTPUT_ROOT / "plans" / f"production_run_plan_{RUNTIME_PLAN['plan_sha256']}.json"
PLAN_BYTES = (json.dumps(RUNTIME_PLAN, sort_keys=True, indent=2, allow_nan=False) + "\n").encode("utf-8")
IMMUTABLE_PAIRS = (
    (CODE_BINDING_PATH, Path(DRIVE_PATHS["output_root"]) / "runtime_code_binding" / RUNTIME_PLAN["plan_sha256"] / f"{ACTUAL_GIT_COMMIT}.json", CODE_BINDING_BYTES),
    (PLAN_PATH, Path(DRIVE_PATHS["output_root"]) / "plans" / f"production_run_plan_{RUNTIME_PLAN['plan_sha256']}.json", PLAN_BYTES),
)
for _path, _logical, _payload in IMMUTABLE_PAIRS:
    if _path.exists() and _path.read_bytes() != _payload:
        raise ValueError(f"Existing local staging immutable differs; preserved: {_path}")
def _remote_immutable_exists(_logical):
    try:
        return STORAGE_ADAPTER.exact_file(_logical) is not None
    except FileNotFoundError:
        return False
PENDING_IMMUTABLE_BYTES = sum(
    len(_payload) for _path, _logical, _payload in IMMUTABLE_PAIRS
    if not _remote_immutable_exists(_logical)
)
if NEXT_PENDING_RECORD is not None:
    require_available_bytes(
        OUTPUT_ROOT,
        required_bytes=int(STORAGE_PROJECTION["drive"]["next_run_required_without_reserve_bytes"]) + PENDING_IMMUTABLE_BYTES,
        reserve_bytes=DEFAULT_STORAGE_RESERVE_BYTES,
        available_bytes=CAPACITY["drive_available_bytes"],
        storage_kind="canonical Drive next-run and immutable plan capacity",
    )
def _publish_immutable(_path, _logical, _payload):
    _path.parent.mkdir(parents=True, exist_ok=True)
    if not _path.exists():
        with _path.open("xb") as _stream:
            _stream.write(_payload)
            _stream.flush()
            os.fsync(_stream.fileno())
    elif _path.read_bytes() != _payload:
        raise ValueError(f"Existing immutable local staging file differs; preserved: {_path}")
    _check = STORAGE_STAGE_ROOT / ".immutable_readback" / hashlib.sha256(str(_logical).encode()).hexdigest()
    try:
        STORAGE_ADAPTER.download_file(_logical, _check)
    except FileNotFoundError:
        _published = STORAGE_ADAPTER.publish_file(_path, _logical)
        if _published.get("status") != "PASS":
            raise RuntimeError(f"Immutable Drive publication did not verify: {_logical}")
        STORAGE_ADAPTER.download_file(_logical, _check)
    if _check.read_bytes() != _payload:
        raise ValueError(f"Immutable Drive read-back differs; preserved both copies: {_logical}")
    _check.unlink(missing_ok=True)
for _path, _logical, _payload in IMMUTABLE_PAIRS:
    _publish_immutable(_path, _logical, _payload)

MATRIX_LEASE_LOGICAL_PATH = LOGICAL_OUTPUT_ROOT / ".matrix_execution.lease.json"
MATRIX_LEASE_PROVISION = STORAGE_ADAPTER.ensure_lease_record(MATRIX_LEASE_LOGICAL_PATH)

drive_available = CAPACITY["drive_available_bytes"]
matrix_peak = int(STORAGE_PROJECTION["drive"]["matrix_peak_with_reserve_bytes"])
matrix_status = "CAPACITY_UNKNOWN" if drive_available is None else (
    "PROJECTED_FIT" if drive_available >= matrix_peak else "INSUFFICIENT_FOR_FULL_MATRIX"
)
print(json.dumps({
    "status": "CANONICAL_STORAGE_READY_NEXT_RUN_GATED",
    "storage_account": CAPACITY["quota_account_email"],
    "canonical_revision_folder_id": CAPACITY["canonical_revision_folder_id"],
    "output_folder_owner": CAPACITY["output_destination"]["destination_owner"],
    "checkpoint_folder_owner": CAPACITY["checkpoint_destination"]["destination_owner"],
    "output_new_file_owner": CAPACITY["output_destination"]["new_file_owner"],
    "checkpoint_new_file_owner": CAPACITY["checkpoint_destination"]["new_file_owner"],
    "quota_status": CAPACITY["quota_status"], "drive_available_bytes": drive_available,
    "next_pending_run": None if NEXT_PENDING_RECORD is None else NEXT_PENDING_RECORD["run_key"],
    "next_run_peak_with_reserve_bytes": STORAGE_PROJECTION["drive"]["next_run_peak_with_reserve_bytes"],
    "full_matrix_projection_status_informational_only": matrix_status,
    "full_matrix_peak_with_reserve_bytes": matrix_peak,
    "completed_run_1": "FINALIZED_COMPLETED_RUN_REUSED_AND_SKIPPED",
    "run_1_measurement_recomputed": False, "run_1_registry_modified": False,
    "training_source_sha256": ACTUAL_TRAINING_SOURCE_SHA256,
    "selected_git_commit": ACTUAL_GIT_COMMIT,
    "schedule_sha256": EXPECTED_SCHEDULE_SHA256,
    "plan_sha256": RUNTIME_PLAN["plan_sha256"],
    "runtime_plan_path": str(PLAN_PATH), "code_binding_path": str(CODE_BINDING_PATH),
    "checkpoint_policy": "local recovery every 5 minutes; Drive every 15 minutes plus task boundaries/interruption; two verified recovery generations; task boundaries retained separately",
    "durable_write_path": "OAuth API with local staging only; DriveFS is not used for durable output",
    "matrix_lease_record": MATRIX_LEASE_PROVISION,
}, sort_keys=True, indent=2))


CELL_C_CANONICAL_COMPLETION_REGISTRY: {"completed_run_keys": ["ember_seed42_full_hybrid"], "drive_file_id": "1hwZai-HEA3dugT1LNCTvsjBBy4Kyjx3-", "drive_parent_id": "10dG3R9EUj28Gc6zJ7LSl-jgn2oKgs5m2", "local_staging_path": "/content/revision_R1_storage_staging/production/results/registry/3941e67baa426cb3caa8a5a0c505d458ed54212934d77f94991eb3dca6148917/matrix_completion_registry.json", "logical_drive_path": "/content/drive/MyDrive/ReVision/revision_R1/production/results/registry/3941e67baa426cb3caa8a5a0c505d458ed54212934d77f94991eb3dca6148917/matrix_completion_registry.json", "plan_sha256": "3941e67baa426cb3caa8a5a0c505d458ed54212934d77f94991eb3dca6148917", "registry_bytes": 5784, "registry_md5": "b9a69fed5a91faf8ff93b1bd20b78e7a"}
{
  "canonical_revision_folder_id": "1tQgfvJNt5MX4cNQbH-4U30hJpcDrBO8x",
  "checkpoint_folder_owner": "kian.anb777@gmail.com",
  "checkpoint_new_file_owner": "kian.anb777@gmail.com",
  "checkpoint_policy": "local recovery every 5 minutes; Drive every 15 minut

In [10]:
# Cell D — read-only Drive diagnostics; save the report locally.
import hashlib
import json
import os
import socket
import sys
import time
import uuid
from datetime import datetime, timezone

from revision_r1.resources import read_host_memory_snapshot
from scripts.production_workflow import (
    _expected_plan_runner_identity,
    inspect_lease_readonly,
    load_run_progress,
    run_checkpoint_root,
    run_output_root,
)
from scripts.live_progress import ColabProgressDisplay

if not globals().get("COLAB_RUNTIME_INSTANCE_ID"):
    COLAB_RUNTIME_INSTANCE_ID = uuid.uuid4().hex

DEBUG_LOGGING = False
live_progress = ColabProgressDisplay(
    interval_seconds=7.0, debug_logging=DEBUG_LOGGING,
)


def _api_stage_optional(logical_path, local_path):
    try:
        STORAGE_ADAPTER.download_file(logical_path, local_path)
        return True
    except FileNotFoundError:
        return False


def _lease_report(logical_path, local_path):
    if _api_stage_optional(logical_path, local_path):
        return inspect_lease_readonly(local_path)
    return {
        "path": str(logical_path),
        "status": "missing",
        "source": "canonical_main_account_drive_api",
        "owner_process_visibility": "unknown",
    }


RUNTIME_FACTS = {
    "runtime_instance_id": COLAB_RUNTIME_INSTANCE_ID,
    "captured_utc": datetime.now(timezone.utc).isoformat(),
    "pid": os.getpid(),
    "hostname": socket.gethostname(),
    "python_executable": sys.executable,
    "python_version": sys.version,
    "memory": read_host_memory_snapshot(),
    "remote_colab_sessions": "not observable from Drive API; check Colab Manage Sessions",
    "mount_account_used_for_durable_writes": False,
}

MATRIX_LEASE_LOCAL = OUTPUT_ROOT / ".matrix_execution.lease.json"
MATRIX_LEASE = _lease_report(
    LOGICAL_OUTPUT_ROOT / MATRIX_LEASE_LOCAL.name,
    MATRIX_LEASE_LOCAL,
)

FIRST_RUN = RUNTIME_PLAN["runs"][0]
FIRST_RUN_LOCK_LOCAL = run_output_root(OUTPUT_ROOT, FIRST_RUN) / ".run.lease.json"
FIRST_RUN_LOCK = _lease_report(
    LOGICAL_OUTPUT_ROOT / FIRST_RUN_LOCK_LOCAL.relative_to(OUTPUT_ROOT),
    FIRST_RUN_LOCK_LOCAL,
)

NEXT_RUN_LOCK = None
NEXT_RUN_PROGRESS = None
CHECKPOINT_METADATA = []

if NEXT_PENDING_RECORD is not None:
    _next_output_local = run_output_root(OUTPUT_ROOT, NEXT_PENDING_RECORD)
    _next_lock_local = _next_output_local / ".run.lease.json"
    NEXT_RUN_LOCK = _lease_report(
        LOGICAL_OUTPUT_ROOT / _next_lock_local.relative_to(OUTPUT_ROOT),
        _next_lock_local,
    )

    _progress_dir = OUTPUT_ROOT / "progress" / RUNTIME_PLAN["plan_sha256"]
    _progress_logical = LOGICAL_OUTPUT_ROOT / "progress" / RUNTIME_PLAN["plan_sha256"]
    try:
        _progress_children = STORAGE_ADAPTER.list_files(_progress_logical)
    except FileNotFoundError:
        _progress_children = []

    _progress_dir.mkdir(parents=True, exist_ok=True)
    for _item in _progress_children:
        if _item.get("mimeType") != "application/vnd.google-apps.folder":
            STORAGE_ADAPTER.download_file(
                _progress_logical / _item["name"],
                _progress_dir / _item["name"],
            )

    _next_progress_path = _progress_dir / (
        f"{NEXT_PENDING_RECORD['run_key']}_{NEXT_PENDING_RECORD['config_sha256'][:12]}.json"
    )
    NEXT_RUN_PROGRESS = load_run_progress(
        _next_progress_path, RUNTIME_PLAN, NEXT_PENDING_RECORD,
    )

    _checkpoint_local = run_checkpoint_root(CHECKPOINT_ROOT, NEXT_PENDING_RECORD) / "recovery"
    _checkpoint_logical = LOGICAL_CHECKPOINT_ROOT / _checkpoint_local.relative_to(CHECKPOINT_ROOT)
    try:
        _checkpoint_children = STORAGE_ADAPTER.list_files(_checkpoint_logical)
    except FileNotFoundError:
        _checkpoint_children = []

    _manifest_cache = OUTPUT_ROOT / "diagnostics" / "checkpoint_manifests"
    _manifest_cache.mkdir(parents=True, exist_ok=True)
    _expected_identity = _expected_plan_runner_identity(RUNTIME_PLAN, NEXT_PENDING_RECORD)

    for _item in _checkpoint_children:
        _name = str(_item.get("name", ""))
        if not _name.endswith(".manifest.json"):
            continue

        _local_manifest = _manifest_cache / _name
        STORAGE_ADAPTER.download_file(_checkpoint_logical / _name, _local_manifest)
        _manifest_bytes = _local_manifest.read_bytes()
        _manifest = json.loads(_manifest_bytes)
        _payload_name = _manifest.get("payload_file")
        _payload_meta = next(
            (row for row in _checkpoint_children if row.get("name") == _payload_name),
            None,
        )
        _identity_matches = _manifest.get("identity") == _expected_identity
        _size_matches = (
            isinstance(_payload_meta, dict)
            and _payload_meta.get("size") is not None
            and _manifest.get("payload_bytes") is not None
            and str(_payload_meta["size"]) == str(_manifest["payload_bytes"])
        )
        CHECKPOINT_METADATA.append({
            "generation": _manifest.get("generation"),
            "kind": _manifest.get("kind"),
            "manifest_file_id": _item.get("id"),
            "manifest_sha256_local_readback": hashlib.sha256(_manifest_bytes).hexdigest(),
            "identity_matches_plan": _identity_matches,
            "payload_name": _payload_name,
            "payload_file_id": None if _payload_meta is None else _payload_meta.get("id"),
            "payload_size_matches_manifest": _size_matches,
            "payload_drive_md5_present": bool(_payload_meta and _payload_meta.get("md5Checksum")),
            "payload_sha256_rechecked": False,
            "status": (
                "METADATA_CONSISTENT_NOT_FULLY_HASH_VERIFIED"
                if _identity_matches and _size_matches
                else "MISMATCH_OR_INCOMPLETE"
            ),
        })

LEASE_RAM_DIAGNOSTIC = {
    "status": "READ_ONLY_DIAGNOSTIC",
    "runtime": RUNTIME_FACTS,
    "storage_identity": {
        "expected_main_account": STORAGE_ACCOUNT_EMAIL,
        "verified_api_account": STORAGE_SESSION_EVIDENCE["storage_account_email"],
        "compute_mount_identity_assumed": False,
        "durable_backend": "explicit_main_account_drive_api_with_local_staging",
        "write_probe": STORAGE_SESSION_EVIDENCE.get("write_probe"),
        "conditional_lease_update_probe": STORAGE_SESSION_EVIDENCE.get("lease_compare_and_swap"),
    },
    "run_1": {"status": "FINALIZED_AND_SKIPPED", "lease": FIRST_RUN_LOCK},
    "next_pending_run": None if NEXT_PENDING_RECORD is None else NEXT_PENDING_RECORD["run_key"],
    "matrix_lease": MATRIX_LEASE,
    "next_run_lease": NEXT_RUN_LOCK,
    "next_run_progress": NEXT_RUN_PROGRESS,
    "next_run_recovery_checkpoint_metadata_only": CHECKPOINT_METADATA,
    "checkpoint_sha256_validation": (
        "Runner verifies full payload SHA before resume; this cell reads metadata only."
    ),
    "no_training_or_lock_recovery": True,
}

DIAGNOSTIC_DIR = OUTPUT_ROOT / "diagnostics"
DIAGNOSTIC_DIR.mkdir(parents=True, exist_ok=True)
LEASE_DIAGNOSTIC_PATH = DIAGNOSTIC_DIR / (
    f"storage_lease_diagnostic_{RUNTIME_PLAN['plan_sha256'][:12]}_{time.time_ns()}.json"
)
with LEASE_DIAGNOSTIC_PATH.open("x", encoding="utf-8") as _stream:
    json.dump(
        LEASE_RAM_DIAGNOSTIC, _stream,
        sort_keys=True, indent=2, default=str, allow_nan=False,
    )
    _stream.flush()
    os.fsync(_stream.fileno())

# The report is optional; leave it in local staging instead of uploading it.
print("Diagnostic saved locally:", LEASE_DIAGNOSTIC_PATH)
print("Matrix lease:", MATRIX_LEASE["status"])
print("Next pending run:", LEASE_RAM_DIAGNOSTIC["next_pending_run"])
print("Next run lease:", NEXT_RUN_LOCK["status"] if NEXT_RUN_LOCK is not None else "N/A")
print("Run 1: finalized and skipped.")
print("Checkpoint payload hashes will be verified before resume.")
if DEBUG_LOGGING:
    print(json.dumps(LEASE_RAM_DIAGNOSTIC, sort_keys=True, indent=2, default=str))
print("No lock was changed or recovered. No training was launched.")

Diagnostic saved locally: /content/revision_R1_storage_staging/production/results/diagnostics/storage_lease_diagnostic_3941e67baa42_1794042064780721020.json
Matrix lease: inactive
Next pending run: ember_seed42_current_only
Next run lease: inactive
Run 1: finalized and skipped.
Checkpoint payload hashes will be verified before resume.
No lock was changed or recovered. No training was launched.


In [12]:
# Cell E — deliberate no-op: Run 1 is already finalized and registered.
import json
print("ALREADY_FINALIZED_COMPLETED_RUN — Run 1 was identity-validated and its existing storage measurement reused in Cell C.")
print("This cell will not train, load a model, remeasure, re-register, or rerun finalization for Run 1.")
if globals().get("DEBUG_LOGGING", False):
    print("CELL_E_REUSES_CELL_C_REGISTRY_EVIDENCE:", json.dumps(MATRIX_REGISTRY_EVIDENCE, sort_keys=True))


ALREADY_FINALIZED_COMPLETED_RUN — Run 1 was identity-validated and its existing storage measurement reused in Cell C.
This cell will not train, load a model, remeasure, re-register, or rerun finalization for Run 1.


In [ ]:
# Cell F is rendered by scripts/build_github_handoff.py from the pinned production runner.
raise RuntimeError("Use the generated, pinned production Cell F from this notebook release.")


In [ ]:
# Cell G — stage compact exports locally, then publish every durable export through canonical OAuth API.
import json, os, shutil, time
from scripts.production_workflow import export_production_progress
from revision_r1.storage import require_available_bytes, DEFAULT_STORAGE_RESERVE_BYTES

EXPORT_CHECKPOINT_BACKUP = False
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
_progress_snapshot = production_workflow_module.inspect_drive_destination(
    DRIVE_PATHS["export_root"], api_factory=STORAGE_API_FACTORY,
    storage_adapter=STORAGE_ADAPTER,
)
require_available_bytes(
    EXPORT_ROOT, required_bytes=1 << 20, reserve_bytes=DEFAULT_STORAGE_RESERVE_BYTES,
    available_bytes=_progress_snapshot["available_bytes"], storage_kind="canonical API progress export",
)
PROGRESS_REPORT = export_production_progress(RUNTIME_PLAN, OUTPUT_ROOT)
PROGRESS_PATH = EXPORT_ROOT / f"production_progress_{RUNTIME_PLAN['plan_sha256'][:12]}_{time.time_ns()}.json"
with PROGRESS_PATH.open("x", encoding="utf-8") as _stream:
    json.dump(PROGRESS_REPORT, _stream, sort_keys=True, indent=2, allow_nan=False)
    _stream.flush(); os.fsync(_stream.fileno())
PROGRESS_PUBLICATION = STORAGE_ADAPTER.publish_file(
    PROGRESS_PATH, Path(DRIVE_PATHS["export_root"]) / PROGRESS_PATH.name,
)
print("Progress export API read-back verified:", json.dumps(PROGRESS_PUBLICATION, sort_keys=True))
print("Registered completions:", PROGRESS_REPORT["validated_completion_count"], "/ 100")

if PROGRESS_REPORT["validated_completion_count"]:
    _completed = [row for row in RUNTIME_PLAN["runs"] if row["run_key"] in MATRIX_REGISTRY["completed_runs"]]
    _remote_output_bytes = sum(
        STORAGE_ADAPTER.tree_bytes(
            Path(DRIVE_PATHS["output_root"]) / run_output_root(OUTPUT_ROOT, _row).relative_to(OUTPUT_ROOT)
        ) for _row in _completed
    )
    require_available_bytes(
        OUTPUT_ROOT, required_bytes=_remote_output_bytes,
        reserve_bytes=DEFAULT_STORAGE_RESERVE_BYTES,
        available_bytes=shutil.disk_usage(OUTPUT_ROOT.parent).free,
        storage_kind="local staging for lightweight result inputs",
    )
    from scripts.production_workflow import _hydrate_api_run_state, run_checkpoint_root
    for _row in _completed:
        _hydrate_api_run_state(
            STORAGE_ADAPTER, OUTPUT_ROOT, RUNTIME_PLAN, _row,
            run_output_root(OUTPUT_ROOT, _row), run_checkpoint_root(CHECKPOINT_ROOT, _row),
            include_recovery=False,
        )
    _results_snapshot = production_workflow_module.inspect_drive_destination(
        DRIVE_PATHS["export_root"], api_factory=STORAGE_API_FACTORY,
        storage_adapter=STORAGE_ADAPTER,
    )
    _results_path = EXPORT_ROOT / f"revision_R1_lightweight_results_{RUNTIME_PLAN['plan_sha256'][:12]}_{time.time_ns()}.zip"
    RESULTS_EXPORT = production_workflow_module.export_lightweight_results(
        RUNTIME_PLAN, OUTPUT_ROOT, _results_path, storage_snapshot=_results_snapshot,
        storage_adapter=STORAGE_ADAPTER, logical_output_path=DRIVE_PATHS["export_root"],
    )
    RESULTS_PUBLICATION = STORAGE_ADAPTER.publish_file(
        _results_path, Path(DRIVE_PATHS["export_root"]) / _results_path.name,
    )
    RESULTS_SIDECAR_PUBLICATION = STORAGE_ADAPTER.publish_file(
        Path(str(_results_path) + ".sha256"),
        Path(DRIVE_PATHS["export_root"]) / (_results_path.name + ".sha256"),
    )
    print("Lightweight results only; API-verified ZIP and sidecar:",
          json.dumps({"archive": RESULTS_PUBLICATION, "sidecar": RESULTS_SIDECAR_PUBLICATION}, sort_keys=True))
else:
    print("No registered completion; no results archive created.")

if EXPORT_CHECKPOINT_BACKUP:
    raise RuntimeError(
        "Checkpoint backup remains intentionally disabled in this handoff; it requires a separately capacity-gated "
        "full checkpoint download and is not needed for resume. No checkpoint copy was started."
    )
print("Full checkpoint archive: NOT_REQUESTED. No historical checkpoint was changed or deleted.")
